## INIT


In [0]:
import sys
import os
sys.path.append(os.path.abspath('../../'))
from utils.logger_utils import get_project_logger
from datetime import datetime
from utils.pipeline_monitoring import log_pipeline_run
from utils.fact_monotoring import pipeline_run

dataset_name = "ev_count_per_area_fact"
layer = "gold"

logger = get_project_logger("Gold_fact_ev_counts_by_area")
logger.info("--- Starting Gold Layer: Creating fact_ev_counts_by_area ---")

## Creating fact view

In [0]:
start_time = datetime.now()
try:
    logger.info("Step 1/3: Running SQL to create fact_ev_counts_by_area view")
    
    sql_query = """
    CREATE OR REPLACE VIEW transport_lakehouse.gold.fact_ev_counts_by_area AS
    SELECT
    o.ownership_key,
    e.ownership,
    e.district_nm,
    e.region_nm,
    e.vehicle_count
    FROM transport_lakehouse.silver.silver_vehicles_electric e
    LEFT JOIN transport_lakehouse.gold.dim_ownership o
    ON e.ownership = o.ownership

    ORDER BY o.ownership_key
     """
    
    spark.sql(sql_query)
    logger.info("Step 1/3: View created successfully.")
    logger.info("--Performing Quality Check (Row Count)--")
    fact_count = spark.table("transport_lakehouse.gold.fact_ev_counts_by_area").count()
    logger.info(f"Quality Check Passed: fact_ev_counts_by_area contains {fact_count} rows.")
    end_time = datetime.now()

    log_pipeline_run(
        spark=spark,
        dataset_name=dataset_name,
        layer=layer,
        run_start_time=start_time,
        run_end_time=end_time,
        status="SUCCESS",
        rows_ingested=fact_count,
        error_message=None
    )

except Exception as e:
    end_time = datetime.now()
    logger.error(f"Failed to create Gold Fact: {str(e)}")

    log_pipeline_run(
        spark=spark,
        dataset_name=dataset_name,
        layer=layer,
        run_start_time=start_time,
        run_end_time=end_time,
        status="FAILED",
        rows_ingested=0,
        error_message=str(e)
    )

    raise

    logger.info("--- Gold fact_ev_counts_by_area Process Completed ---")

## Missing keys check

In [0]:
logger.info("Step 2/3: Starting missing keys check...")

df_check = spark.sql("""
SELECT
  SUM(CASE WHEN ownership_key IS NULL THEN 1 ELSE 0 END) AS missing_ownership
FROM transport_lakehouse.gold.fact_ev_counts_by_area
""")

results = df_check.collect()[0]
missing_keys_results =  {"ownership ": results['missing_ownership']}

missing_report = (
    f"Missing Keys Summary: "
    f"Ownership: {results['missing_ownership']}"
)
missing_keys_report = results

if results['missing_ownership'] > 0:
    logger.warning(f"Step 2/3: Finished with MISSING KEYS! {missing_report}")
else:
    logger.info(f"Step 2/3: Finished missing keys check successfully. All keys are mapped. {missing_report}")


## Key null ratio check

In [0]:
logger.info("Step 3/3: Starting Key null ratio check...")

df_check = spark.sql("""
SELECT
  CAST(AVG(CASE WHEN ownership_key IS NOT NULL THEN 1 ELSE 0 END) AS FLOAT) AS ownership_key_fill_rate
FROM transport_lakehouse.gold.fact_ev_counts_by_area
""")

results = df_check.collect()[0]
fill_rate_results = {"ownership ": results['ownership_key_fill_rate']}

missing_report = (
    f"Key null ratio Summary: "
    f"Ownership: {results['ownership_key_fill_rate']}"
)

if results['ownership_key_fill_rate'] < 1:
    logger.warning(f"Step 3/3: Finished with MISSING KEYS! {missing_report}")
else:
    logger.info(f"Step 3/3: Finished  Key null ratio successfully. All keys are mapped. {missing_report}")


## Fact monotoring

In [0]:
logger.info("Starting fact monitoring...")
pipeline_run(
    spark,
    dataset_name=dataset_name,
    layer=layer,
    missing_keys = missing_keys_results,
    duplication_check = False,
    key_null_ratio = fill_rate_results
)
logger.info("Completed fact monitoring...")